## Mapping

In [ ]:
import pandas as pd
from src.config import *  
import re 

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
#Load metadata and raw data
survey = pd.read_excel(DIR_META_DATA, sheet_name='survey')
choices = pd.read_excel(DIR_META_DATA, sheet_name='choices')
raw_data=pd.read_csv(DIR_CLEANED_DATA)

In [ ]:
# Read the manual QA review
qa = pd.read_csv(DIR_QA)

In [ ]:
len(raw_data)

### Sanity Checks

In [ ]:
meta_columns=survey.name.tolist()
data_columns=raw_data.columns.tolist()

missing_in_meta=[col for col in data_columns if col not in meta_columns]
if missing_in_meta:
    print("Columns in raw data missing in metadata:", missing_in_meta)
missing_in_data=[col for col in meta_columns if col not in data_columns]
if missing_in_data:
    print("Columns in metadata missing in raw data:", missing_in_data)

### Prepare data for mapping
- convert columns to be mapped to string
- create mapping dictionary from choices dataframe


In [ ]:
def normalize_column(s: pd.Series) -> pd.Series:
    dtype = s.dtype

    if pd.api.types.is_float_dtype(dtype):
        return s.astype('Int64').astype('string')
    if pd.api.types.is_integer_dtype(dtype):
        return s.astype('Int64').astype('string')
    if pd.api.types.is_object_dtype(dtype) or pd.api.types.is_string_dtype(dtype):
        return s.astype('string').replace({'nan': pd.NA, '': pd.NA})
    
    return s


In [ ]:
mapping = {}
for list_name, group in choices.groupby('list_name'):
    mapping[list_name] = dict(zip(group['value'], group['label']))

In [ ]:
# Extract list names from 'type' column for select_one and select_multiple questions
selection_mask=survey['type'].str.startswith('select')
survey_select=survey.loc[selection_mask,["type","name","label", "hint"]].rename(columns={"name":"col_name"})
# Get the choices list names from type column
# (list_name can include digits, e.g. "likert_scale_10" -- don't drop them)
pattern = r'select_[a-z]*\s+([a-z0-9_]+)'
survey_select["list_name"]=survey_select['type'].apply(lambda x: re.search(pattern, str(x)).group(1))


In [ ]:
survey_select=survey_select[survey_select["col_name"].isin(raw_data.columns)]

In [ ]:
# Convert all mapping columns to string
cols_to_map=survey_select["col_name"].tolist()
for col in cols_to_map:
    raw_data[col]=normalize_column(raw_data[col])

### Map "other" radio stations
Free-text `station_*_other` answers (all three station questions) are split into one entry per station by `src/stations.py`: the reviewed lookup `DIR_STATIONS_OTHER_CLEANED` first, then known-station patterns, then fuzzy matching against known names and the questionnaire's choices; anything unmatched is kept as typed. Each station keeps the FM frequency written next to it; a mention without one takes the frequency of the nearest respondent (by GPS) who named the same station with one, if they are within 10 km (`FREQ_IMPUTE_MAX_KM`). "Didn't listen" answers become `-99`. Names matching a questionnaire choice become that choice's code. The entries then replace the `-22` (other) code in each station column, so every station is counted separately.

Review `4_validation_checks/01_station_other_standardised.csv` (every answer, how it was read and what it became; check `typed` and `fuzzy` first) and `01_station_frequencies.csv` (each station/frequency, with how many frequencies were imputed and from how far away).

In [ ]:
from src.stations import (NO_LISTENING, StationStandardiser, format_entry, impute_frequencies,
                          station_key)

def _code(v):
    return str(int(v)) if isinstance(v, float) and v.is_integer() else str(v).strip()

# Questionnaire station choices: a standardised name that matches a choice label becomes that choice's
# code, so it's counted together with respondents who ticked that station.
station_list = survey.loc[survey["name"] == "station_most_listened", "type"].iloc[0].split()[-1]
station_choices = choices.loc[choices["list_name"] == station_list]
choice_by_key = {station_key(label): _code(value)
                 for value, label in zip(station_choices["value"], station_choices["label"])
                 if not _code(value).startswith("-")}   # skip other / don't know / refused

# Reviewed lookup (raw answer -> "Name freq; Name"), then patterns, fuzzy matching and "kept as typed"
stations_other = pd.read_csv(DIR_STATIONS_OTHER_CLEANED, dtype=str, keep_default_na=False)
std = StationStandardiser(
    lookup=dict(zip(stations_other["station_most_listened_other"], stations_other["station_most_listened_other_cleaned"])),
    extra_names=station_choices.loc[~station_choices["value"].map(_code).str.startswith("-"), "label"],
)

# One row per station mentioned, per respondent and question (an *_other column with no answers at all
# was dropped in 00 as 100% missing)
station_other_cols = [c for c in ["station_most_listened_other", "station_listened_yesterday_other",
                                  "station_past_7_days_other"] if c in raw_data.columns]
rows = []
for col in station_other_cols:
    for idx, answer in raw_data[col].dropna().items():
        entries, method = std.standardise(answer)
        rows += [dict(idx=idx, col=col, answer=answer, method=method, name=n, freq=f) for n, f in entries]
mentions = pd.DataFrame(rows, columns=["idx", "col", "answer", "method", "name", "freq"])
mentions = mentions.join(raw_data[["gps-Latitude", "gps-Longitude"]], on="idx")

# A mention without a frequency takes the one given by the nearest respondent (GPS) naming the same station,
# if that respondent is within FREQ_IMPUTE_MAX_KM; otherwise it stays unknown
FREQ_IMPUTE_MAX_KM = 10
mentions = impute_frequencies(mentions, lat_col="gps-Latitude", lon_col="gps-Longitude", max_km=FREQ_IMPUTE_MAX_KM)
mentions["code"] = mentions["name"].map(lambda n: choice_by_key.get(station_key(n)))
mentions["entry"] = [n if n == NO_LISTENING else code if code else format_entry(n, f)
                     for n, f, code in zip(mentions["name"], mentions["freq"], mentions["code"])]

print(f"{mentions['answer'].nunique()} distinct 'other' answers -> {len(mentions)} station mentions")
print("How answers were standardised:", mentions.drop_duplicates(['idx', 'col'])['method'].value_counts().to_dict())
print(f"Frequencies: {mentions['freq'].notna().sum() - mentions['freq_imputed'].sum()} stated, "
      f"{mentions['freq_imputed'].sum()} taken from the nearest respondent, "
      f"{(mentions['freq'].isna() & mentions['code'].isna() & ~mentions['name'].isin([NO_LISTENING])).sum()} unknown "
      f"(of which {(mentions['freq'].isna() & (mentions['freq_donor_km'] > FREQ_IMPUTE_MAX_KM)).sum()} because the "
      f"nearest respondent with a frequency was more than {FREQ_IMPUTE_MAX_KM} km away)")

# Review files (answers and station names only, no respondent rows)
(mentions.groupby(["answer", "method"]).agg(standardised=("entry", lambda s: "; ".join(dict.fromkeys(s))), n=("idx", "nunique"))
    .reset_index().sort_values(["method", "n"], ascending=[False, False])
    .to_csv(DIR_VALIDATE/"01_station_other_standardised.csv", index=False, encoding="utf-8-sig"))
(mentions.groupby(["name", "freq"], dropna=False)
    .agg(mentions=("idx", "size"), imputed=("freq_imputed", "sum"), max_donor_km=("freq_donor_km", "max"),
         choice_code=("code", "first"))
    .reset_index().round(1)
    .to_csv(DIR_VALIDATE/"01_station_frequencies.csv", index=False, encoding="utf-8-sig"))

# Each *_other column becomes ", "-separated choice codes / "Name freq" entries for the combine step below
for col in station_other_cols:
    joined = mentions[mentions["col"] == col].groupby("idx")["entry"].agg(lambda s: ", ".join(dict.fromkeys(s)))
    raw_data[col] = joined.reindex(raw_data.index)

In [ ]:
# Put the mapped "other" answers into the station columns, in place of the -22 (other) code.
# 1) Replace spaces inside tokens with underscores. Keep ", " as delimiter.
def to_underscored_space_sep(val):
    if pd.isna(val): return val
    parts = re.split(r"\s*,\s*", str(val).strip())
    parts = [re.sub(r"\s+", "_", p.strip()) for p in parts if p.strip()]
    return ", ".join(parts)

# 2) Remove commas.
def remove_commas(val):
    if pd.isna(val): return val
    return str(val).replace(",", "")

# 3) Combine main with *_other, applying rules for -22, -88, -99.
def _combine_value(main_val, other_val, other_code="-22", miss_codes=("-88","-99")):
    if pd.isna(main_val): return main_val
    s = str(main_val).strip()
    if s in miss_codes: return s
    tokens = re.split(r"\s+", s)
    out = []
    for t in tokens:
        if t in miss_codes:       # drop -88/-99 when mixed with values
            continue
        if t == other_code:       # replace -22 with other
            if pd.notna(other_val) and str(other_val).strip():
                out.append(str(other_val).strip())
            continue
        out.append(t)
    # de-duplicate: an "other" answer mapped to a station the respondent also ticked counts once
    res = " ".join(dict.fromkeys(" ".join(out).split()))
    if not res and pd.notna(other_val):  # only -22 case
        return str(other_val).strip()
    return res

def combine_columns(df, main_col, other_col, other_code="-22", miss_codes=("-88","-99")):
    df[main_col] = df.apply(lambda r: _combine_value(r[main_col], r[other_col], other_code, miss_codes), axis=1)
    return df

def normalize_88_col(s):
    return s.astype("string").str.replace(r"^\s*88\s*$", "-88", regex=True)

# Usage (an *_other column with no answers at all was dropped in 00 as 100% missing)
cols = ["station_most_listened", "station_listened_yesterday", "station_past_7_days"]
for col in cols:
    other = col + "_other"
    if other not in raw_data.columns:
        continue
    raw_data[other] = raw_data[other].apply(to_underscored_space_sep)
    raw_data[other] = raw_data[other].apply(remove_commas)
    combine_columns(raw_data, col, other)
    raw_data[col] = normalize_88_col(raw_data[col])

raw_data = raw_data.drop(columns=[c + "_other" for c in cols], errors="ignore")

### Mapping of the columns

In [ ]:
# Some columns contain lists of values separated by spaces; we need to split them before mapping
list_columns=survey_select[survey_select["type"].str.startswith('select_multiple')]["col_name"].tolist()
list_columns = [col for col in list_columns if col in survey_select["col_name"].tolist()]
single_value_columns = [col for col in cols_to_map if col not in list_columns]



In [ ]:
def map_select_multiple(column: pd.Series, mapping_dict: dict) -> pd.Series:
    def map_values(value: pd.Series) -> pd.Series:
        if pd.isna(value):
            return pd.NA
        else:
            values = str(value).split()
            mapped_values = [mapping_dict.get(v, v) for v in values]
            return '|'.join(mapped_values)
    
    return column.apply(map_values)

In [ ]:
# Apply mapping to all single value columns
mapped_data=raw_data.copy()
survey_select_single=survey_select[survey_select["col_name"].isin(single_value_columns)]
for col_name, list_name in zip(survey_select_single["col_name"],survey_select_single["list_name"]):
    mapped_data.loc[:,col_name]=mapped_data.loc[:,col_name].map(mapping[list_name])


In [ ]:
# Apply mapping to all list columns
survey_select_list=survey_select[survey_select["col_name"].isin(list_columns)]
for col_name, list_name in zip(survey_select_list["col_name"],survey_select_list["list_name"]):
    mapped_data.loc[:,col_name]=map_select_multiple(mapped_data.loc[:,col_name],mapping[list_name])     

###

In [ ]:
cols=["station_most_listened","station_listened_yesterday","station_past_7_days"]
mapped_data[cols] = mapped_data[cols].apply(lambda x: x.str.replace('_', ' '))


#### Correct department and location labels

In [ ]:
# keep raw version just in case
mapped_data['province_old'] = mapped_data['province']
mapped_data['location_old'] = mapped_data['location']

# create dictionary of keys and department names
dpt_dict = qa.set_index([QA_KEY_COL])[QA_PROVINCE_COL].to_dict()

# repeat for location names
# create dictionary of keys and location names
loc_dict = qa.set_index([QA_KEY_COL])[QA_LOCATION_COL].to_dict()


# create functions
def map_province(x):
    try:
        dpt_dict[x]
    except KeyError:
        return None

def map_location(x):
    try:
        loc_dict[x]
    except KeyError:
        return None


# update province column with manual review
mapped_data['province'] = mapped_data['KEY'].map(dpt_dict).fillna(mapped_data['province_old'])

# update location column with manual review
mapped_data['location'] = mapped_data['KEY'].map(loc_dict).fillna(mapped_data['location_old'])


In [ ]:
mapped_data.loc[mapped_data.province_old != mapped_data.province].groupby(['province_old', 'province']).count()

In [ ]:
mapped_data.loc[mapped_data.location_old != mapped_data.location].groupby(['location_old', 'location']).count()

### Export mapped data

In [ ]:
mapped_data.to_csv(DIR_MAPPED_DATA, index=False)